In [11]:
!pip -q install requests beautifulsoup4 PyPDF2 pandas

In [16]:
1

# ================================================================
# 🚀 CAREERPILOT-X 2.0
# AI Career Intelligence & Verified Job Assistant
# Google Colab / Jupyter Notebook Version
# ================================================================

import os
import re
import json
import sqlite3
import hashlib
import secrets
import math
import getpass
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime

try:
    from PyPDF2 import PdfReader
except:
    PdfReader = None


# ================================================================
# 1. CONFIGURATION
# ================================================================

APP_NAME = "CareerPilot-X 2.0"

# Render-safe / notebook-safe storage.
# IMPORTANT: No hard-coded /content dependency.
BASE_DIR = Path.cwd() / "careerpilot_data"
BASE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = BASE_DIR / "careerpilot.db"

print("=" * 70)
print("🚀 CAREERPILOT-X 2.0")
print("=" * 70)
print(f"📁 Data directory: {BASE_DIR}")
print("✅ Storage initialized")
print()


# ================================================================
# 2. DATABASE
# ================================================================

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password_hash TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS profiles (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    college TEXT,
    degree TEXT,
    year TEXT,
    cgpa TEXT,
    skills TEXT,
    interests TEXT,
    target_role TEXT,
    experience TEXT,
    updated_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS applications (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    company TEXT,
    role TEXT,
    url TEXT,
    status TEXT,
    notes TEXT,
    applied_date TEXT,
    FOREIGN KEY(user_id) REFERENCES users(id)
)
""")

conn.commit()


# ================================================================
# 3. PASSWORD SECURITY
# ================================================================

def hash_password(password):
    salt = secrets.token_bytes(16)

    key = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        salt,
        120000
    )

    return salt.hex() + ":" + key.hex()


def verify_password(password, stored):
    try:
        salt_hex, hash_hex = stored.split(":")
        salt = bytes.fromhex(salt_hex)

        key = hashlib.pbkdf2_hmac(
            "sha256",
            password.encode(),
            salt,
            120000
        )

        return secrets.compare_digest(
            key.hex(),
            hash_hex
        )

    except:
        return False


# ================================================================
# 4. LOGIN / SIGNUP
# ================================================================

CURRENT_USER = None


def signup():
    print("\n🔐 CREATE ACCOUNT")
    print("-" * 40)

    username = input("Choose username: ").strip()

    if not username:
        print("❌ Username cannot be empty.")
        return

    password = getpass.getpass("Create password: ")
    confirm = getpass.getpass("Confirm password: ")

    if password != confirm:
        print("❌ Passwords do not match.")
        return

    if len(password) < 6:
        print("❌ Password must contain at least 6 characters.")
        return

    try:
        cursor.execute(
            """
            INSERT INTO users(username, password_hash, created_at)
            VALUES (?, ?, ?)
            """,
            (
                username,
                hash_password(password),
                datetime.now().isoformat()
            )
        )

        conn.commit()

        print("✅ Account created successfully!")

    except sqlite3.IntegrityError:
        print("❌ Username already exists.")


def login():
    global CURRENT_USER

    print("\n🔐 LOGIN")
    print("-" * 40)

    username = input("Username: ").strip()
    password = getpass.getpass("Password: ")

    cursor.execute(
        "SELECT id, username, password_hash FROM users WHERE username=?",
        (username,)
    )

    row = cursor.fetchone()

    if row and verify_password(password, row[2]):
        CURRENT_USER = {
            "id": row[0],
            "username": row[1]
        }

        print(f"\n✅ Welcome back, {row[1]}!")
        return True

    print("❌ Invalid username or password.")
    return False


def authentication():
    global CURRENT_USER

    while CURRENT_USER is None:

        print("\n" + "=" * 60)
        print("🔐 CAREERPILOT-X SECURITY")
        print("=" * 60)
        print("1. Login")
        print("2. Create Account")
        print("3. Continue as Demo User")
        print("4. Exit")

        choice = input("\nChoose: ").strip()

        if choice == "1":
            login()

        elif choice == "2":
            signup()

        elif choice == "3":
            CURRENT_USER = {
                "id": 0,
                "username": "demo_user"
            }
            print("✅ Demo mode activated.")

        elif choice == "4":
            print("👋 Goodbye!")
            return False

        else:
            print("❌ Invalid choice.")

    return True


# ================================================================
# 5. CAREER DATABASE
# ================================================================

CAREERS = {

    "AI/ML Engineer": {
        "skills": [
            "Python", "Machine Learning", "Statistics",
            "NumPy", "Pandas", "Deep Learning",
            "TensorFlow", "PyTorch"
        ],
        "salary": "₹6L - ₹20L+",
        "description": "Build and deploy machine learning and AI systems."
    },

    "Generative AI Engineer": {
        "skills": [
            "Python", "LLM", "Generative AI",
            "RAG", "Prompt Engineering",
            "LangChain", "Vector Databases"
        ],
        "salary": "₹7L - ₹25L+",
        "description": "Build applications using generative AI and LLMs."
    },

    "Data Scientist": {
        "skills": [
            "Python", "Statistics", "Machine Learning",
            "Pandas", "NumPy", "SQL",
            "Data Visualization"
        ],
        "salary": "₹6L - ₹18L+",
        "description": "Extract insights and build predictive models."
    },

    "Data Analyst": {
        "skills": [
            "Python", "SQL", "Excel",
            "Power BI", "Tableau",
            "Statistics"
        ],
        "salary": "₹4L - ₹12L",
        "description": "Analyze business data and create insights."
    },

    "Software Developer": {
        "skills": [
            "Python", "Java", "Data Structures",
            "Algorithms", "Git", "SQL"
        ],
        "salary": "₹4L - ₹15L",
        "description": "Design and develop software applications."
    },

    "Cybersecurity Analyst": {
        "skills": [
            "Networking", "Linux", "Python",
            "Cybersecurity", "Ethical Hacking",
            "SIEM"
        ],
        "salary": "₹5L - ₹18L",
        "description": "Protect systems, networks and data."
    },

    "Cloud Engineer": {
        "skills": [
            "AWS", "Azure", "Docker",
            "Linux", "Networking", "Python"
        ],
        "salary": "₹5L - ₹18L",
        "description": "Build and maintain cloud infrastructure."
    },

    "NLP Engineer": {
        "skills": [
            "Python", "NLP", "Transformers",
            "Machine Learning", "Deep Learning",
            "Hugging Face"
        ],
        "salary": "₹6L - ₹20L+",
        "description": "Build systems that understand human language."
    },

    "AI Product Manager": {
        "skills": [
            "AI", "Product Management",
            "Communication", "Analytics",
            "Agile", "Business"
        ],
        "salary": "₹8L - ₹25L+",
        "description": "Manage AI-powered products and teams."
    }
}


# ================================================================
# 6. STUDENT PROFILE
# ================================================================

def save_profile(profile):

    if CURRENT_USER["id"] == 0:
        print("ℹ️ Demo profile is temporary.")
        return

    cursor.execute("""
    INSERT OR REPLACE INTO profiles
    (
        user_id, name, college, degree, year,
        cgpa, skills, interests, target_role,
        experience, updated_at
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        CURRENT_USER["id"],
        profile["name"],
        profile["college"],
        profile["degree"],
        profile["year"],
        profile["cgpa"],
        ",".join(profile["skills"]),
        profile["interests"],
        profile["target_role"],
        profile["experience"],
        datetime.now().isoformat()
    ))

    conn.commit()


def get_profile():

    if CURRENT_USER["id"] == 0:
        return None

    cursor.execute(
        "SELECT * FROM profiles WHERE user_id=?",
        (CURRENT_USER["id"],)
    )

    row = cursor.fetchone()

    if not row:
        return None

    return {
        "name": row[1],
        "college": row[2],
        "degree": row[3],
        "year": row[4],
        "cgpa": row[5],
        "skills": [x.strip() for x in row[6].split(",") if x.strip()],
        "interests": row[7],
        "target_role": row[8],
        "experience": row[9]
    }


def student_profile():

    print("\n👩‍🎓 STUDENT PROFILE")
    print("=" * 60)

    old = get_profile()

    name = input(
        f"Name [{old['name'] if old else ''}]: "
    ).strip()

    college = input(
        f"College [{old['college'] if old else ''}]: "
    ).strip()

    degree = input(
        f"Degree [{old['degree'] if old else ''}]: "
    ).strip()

    year = input(
        f"Year [{old['year'] if old else ''}]: "
    ).strip()

    cgpa = input(
        f"CGPA [{old['cgpa'] if old else ''}]: "
    ).strip()

    skills_text = input(
        "Skills (comma separated): "
    ).strip()

    interests = input(
        "Career interests: "
    ).strip()

    target = input(
        "Target role: "
    ).strip()

    experience = input(
        "Projects / experience: "
    ).strip()

    profile = {
        "name": name,
        "college": college,
        "degree": degree,
        "year": year,
        "cgpa": cgpa,
        "skills": [
            x.strip() for x in skills_text.split(",")
            if x.strip()
        ],
        "interests": interests,
        "target_role": target,
        "experience": experience
    }

    save_profile(profile)

    print("\n✅ Student profile saved.")


# ================================================================
# 7. CAREER INTELLIGENCE
# ================================================================

def normalize(text):
    return re.sub(
        r"[^a-z0-9+#.\- ]",
        " ",
        str(text).lower()
    )


def career_intelligence():

    profile = get_profile()

    if not profile:

        print("⚠️ Create your Student Profile first.")
        return

    user_skills = set(
        normalize(x) for x in profile["skills"]
    )

    results = []

    for career, data in CAREERS.items():

        career_skills = set(
            normalize(x) for x in data["skills"]
        )

        matched = user_skills.intersection(career_skills)

        score = (
            len(matched) / len(career_skills) * 100
            if career_skills else 0
        )

        results.append({
            "Career": career,
            "Match": round(score, 2),
            "Matched Skills": ", ".join(matched),
            "Salary": data["salary"]
        })

    df = pd.DataFrame(results)
    df = df.sort_values("Match", ascending=False)

    print("\n🧠 CAREER INTELLIGENCE")
    print("=" * 70)
    print(df.to_string(index=False))

    top = df.iloc[0]

    print(
        f"\n🏆 Recommended Career: {top['Career']}"
    )


# ================================================================
# 8. VERIFIED JOB SEARCH
# ================================================================

SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "")


def set_serpapi_key():

    global SERPAPI_KEY

    key = getpass.getpass(
        "Enter SerpApi key (hidden): "
    ).strip()

    SERPAPI_KEY = key

    print("✅ SerpApi key configured for this session.")


def search_jobs():

    global SERPAPI_KEY

    query = input(
        "\n🔎 Enter job search query:\n"
    ).strip()

    if not SERPAPI_KEY:

        print("\n⚠️ SerpApi key not configured.")
        print("Using safe demo opportunities instead.\n")

        return demo_jobs(query)

    try:

        params = {
            "engine": "google",
            "q": query,
            "api_key": SERPAPI_KEY,
            "num": 10
        }

        response = requests.get(
            "https://serpapi.com/search.json",
            params=params,
            timeout=20
        )

        data = response.json()

        results = data.get("organic_results", [])

        if not results:
            print("❌ No results found.")
            return []

        jobs = []

        for item in results:

            jobs.append({
                "title": item.get("title", ""),
                "company": item.get("source", "Unknown"),
                "url": item.get("link", ""),
                "snippet": item.get("snippet", "")
            })

        print("\n🔎 LIVE JOB RESULTS")
        print("=" * 80)

        for i, job in enumerate(jobs, 1):

            print(f"\n{i}. {job['title']}")
            print(f"🏢 {job['company']}")
            print(f"🌐 {job['url']}")
            print(f"📝 {job['snippet'][:250]}")

        return jobs

    except Exception as e:

        print(f"❌ Search error: {e}")
        return []


def demo_jobs(query="AI internship"):

    jobs = [

        {
            "title": "AI/ML Intern",
            "company": "Microsoft",
            "url": "https://careers.microsoft.com/",
            "snippet": "AI and machine learning internship opportunities."
        },

        {
            "title": "Machine Learning Intern",
            "company": "Google",
            "url": "https://careers.google.com/",
            "snippet": "Machine learning and software internship roles."
        },

        {
            "title": "Software Development Intern",
            "company": "Amazon",
            "url": "https://www.amazon.jobs/",
            "snippet": "Software engineering internship opportunity."
        },

        {
            "title": "AI Intern",
            "company": "Infosys",
            "url": "https://www.infosys.com/careers/",
            "snippet": "Technology and AI internship opportunities."
        }
    ]

    for i, job in enumerate(jobs, 1):

        print(
            f"{i}. {job['title']} | "
            f"{job['company']} | "
            f"{job['url']}"
        )

    return jobs


# ================================================================
# 9. URL SECURITY
# ================================================================

SHORTENERS = [
    "bit.ly",
    "tinyurl.com",
    "t.co",
    "goo.gl",
    "is.gd",
    "cutt.ly",
    "shorturl.at"
]

SUSPICIOUS_TLDS = [
    ".xyz",
    ".top",
    ".click",
    ".loan",
    ".work",
    ".gq",
    ".tk",
    ".ml",
    ".cf"
]


def analyze_url(url):

    score = 0
    reasons = []

    url_lower = url.lower().strip()

    if not url_lower.startswith(("http://", "https://")):
        url_lower = "https://" + url_lower

    # Shortener
    if any(domain in url_lower for domain in SHORTENERS):
        score += 20
        reasons.append(
            "URL uses a shortened domain."
        )

    # IP address
    if re.search(
        r"https?://(?:\d{1,3}\.){3}\d{1,3}",
        url_lower
    ):
        score += 30
        reasons.append(
            "URL uses an IP address instead of a normal domain."
        )

    # @ symbol
    if "@" in url_lower:
        score += 25
        reasons.append(
            "URL contains '@', which can hide the real destination."
        )

    # Suspicious TLD
    if any(tld in url_lower for tld in SUSPICIOUS_TLDS):
        score += 20
        reasons.append(
            "Suspicious or commonly abused top-level domain."
        )

    # Punycode
    if "xn--" in url_lower:
        score += 25
        reasons.append(
            "Punycode domain detected."
        )

    # Long URL
    if len(url_lower) > 150:
        score += 10
        reasons.append(
            "Unusually long URL."
        )

    # Suspicious words
    suspicious_words = [
        "verify",
        "urgent",
        "login",
        "payment",
        "reward",
        "free",
        "winner",
        "claim",
        "otp",
        "password",
        "bank"
    ]

    found = [
        word for word in suspicious_words
        if word in url_lower
    ]

    if found:
        score += min(25, len(found) * 5)

        reasons.append(
            "Suspicious keywords: " + ", ".join(found)
        )

    score = min(score, 100)

    if score >= 70:
        level = "HIGH RISK"
    elif score >= 40:
        level = "MEDIUM RISK"
    else:
        level = "LOW RISK"

    return {
        "score": score,
        "level": level,
        "reasons": reasons
    }


# ================================================================
# 10. FAKE JOB DETECTION
# ================================================================

def fake_job_detector():

    print("\n🛡️ FAKE JOB DETECTION")
    print("=" * 60)

    title = input("Job title: ")
    company = input("Company: ")
    description = input(
        "Paste job description:\n"
    )

    url = input(
        "Job URL (optional): "
    ).strip()

    text = normalize(
        title + " " +
        company + " " +
        description
    )

    score = 0
    reasons = []

    payment_words = [
        "pay registration",
        "registration fee",
        "processing fee",
        "security deposit",
        "training fee",
        "pay money",
        "send money"
    ]

    sensitive_words = [
        "otp",
        "cvv",
        "pin",
        "password",
        "bank account",
        "credit card",
        "debit card"
    ]

    urgency_words = [
        "urgent",
        "immediately",
        "today only",
        "limited seats",
        "act now"
    ]

    private_contact = [
        "telegram",
        "whatsapp",
        "personal number"
    ]

    unrealistic = [
        "earn ₹1 lakh",
        "guaranteed job",
        "100% guaranteed",
        "no skills required"
    ]

    for word in payment_words:
        if word in text:
            score += 25
            reasons.append(
                f"Payment request detected: {word}"
            )

    for word in sensitive_words:
        if word in text:
            score += 30
            reasons.append(
                f"Sensitive information requested: {word}"
            )

    for word in urgency_words:
        if word in text:
            score += 10
            reasons.append(
                f"Urgency indicator: {word}"
            )

    for word in private_contact:
        if word in text:
            score += 10
            reasons.append(
                f"Private communication request: {word}"
            )

    for word in unrealistic:
        if word in text:
            score += 15
            reasons.append(
                f"Unrealistic promise: {word}"
            )

    if url:

        url_result = analyze_url(url)

        score += int(
            url_result["score"] * 0.5
        )

        reasons.extend(
            url_result["reasons"]
        )

    score = min(int(score), 100)

    if score >= 70:
        level = "🚨 HIGH RISK"
        action = "Do NOT pay, share OTP/password/bank details, or continue."
    elif score >= 40:
        level = "⚠️ MEDIUM RISK"
        action = "Verify the company and job through its official career website."
    else:
        level = "✅ LOWER RISK"
        action = "Still verify the opportunity through the official company source."

    print("\n" + "=" * 60)
    print("🛡️ THREAT REPORT")
    print("=" * 60)
    print(f"🎯 Scam Risk Score: {score}/100")
    print(f"🚦 Risk Level: {level}")

    print("\n🔍 Indicators:")

    if reasons:
        for reason in dict.fromkeys(reasons):
            print(" •", reason)
    else:
        print(" • No major scam indicators detected.")

    print("\n🛡️ Recommended Action:")
    print(action)

    return score


# ================================================================
# 11. OPPORTUNITY DNA
# ================================================================

def opportunity_dna():

    profile = get_profile()

    if not profile:
        print("⚠️ Create Student Profile first.")
        return

    print("\n🧬 OPPORTUNITY DNA")
    print("=" * 60)

    role = input("Opportunity role: ")
    company = input("Company: ")
    description = input(
        "Paste opportunity description:\n"
    )

    user_skills = set(
        normalize(x)
        for x in profile["skills"]
    )

    combined = normalize(
        role + " " +
        description
    )

    matched = [
        skill
        for skill in user_skills
        if skill and skill in combined
    ]

    score = min(
        100,
        len(matched) * 12 + 20
    )

    print(f"\n🏢 Company: {company}")
    print(f"💼 Role: {role}")
    print(f"🧬 DNA Score: {score}/100")

    print(
        "\n🎯 Matching Skills:",
        ", ".join(matched)
        if matched else "None detected"
    )


# ================================================================
# 12. SKILL MATCHING
# ================================================================

def skill_matching():

    profile = get_profile()

    if not profile:
        print("⚠️ Create Student Profile first.")
        return

    role = input(
        "Enter target career role: "
    ).strip()

    if role not in CAREERS:

        print("\nAvailable roles:")

        for x in CAREERS:
            print(" •", x)

        return

    required = CAREERS[role]["skills"]

    student = [
        normalize(x)
        for x in profile["skills"]
    ]

    matched = []
    missing = []

    for skill in required:

        if normalize(skill) in student:
            matched.append(skill)
        else:
            missing.append(skill)

    score = (
        len(matched) /
        len(required) *
        100
    )

    print("\n🎯 SKILL MATCHING")
    print("=" * 60)

    print(f"Career: {role}")
    print(f"Match: {score:.2f}%")

    print(
        "\n✅ Existing Skills:",
        ", ".join(matched) or "None"
    )

    print(
        "\n📚 Skill Gaps:",
        ", ".join(missing) or "None"
    )


# ================================================================
# 13. RESUME ANALYZER
# ================================================================

def extract_resume_text(path):

    path = Path(path)

    if not path.exists():
        return ""

    if path.suffix.lower() == ".txt":

        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    if path.suffix.lower() == ".pdf" and PdfReader:

        text = ""

        reader = PdfReader(str(path))

        for page in reader.pages:
            text += page.extract_text() or ""

        return text

    return ""


def resume_analyzer():

    print("\n📄 RESUME ANALYZER")
    print("=" * 60)

    path = input(
        "Enter resume file path (.pdf/.txt): "
    ).strip()

    text = extract_resume_text(path)

    if not text:

        print("❌ Could not extract resume text.")
        return

    profile = get_profile()

    keywords = [
        "python",
        "java",
        "machine learning",
        "deep learning",
        "artificial intelligence",
        "sql",
        "nlp",
        "generative ai",
        "rag",
        "tensorflow",
        "pytorch",
        "pandas",
        "numpy",
        "git",
        "streamlit"
    ]

    text_lower = text.lower()

    found = [
        x for x in keywords
        if x in text_lower
    ]

    missing = [
        x for x in keywords
        if x not in text_lower
    ]

    sections = {
        "education": "education" in text_lower,
        "skills": "skills" in text_lower,
        "projects": "projects" in text_lower,
        "experience": "experience" in text_lower,
        "certifications": "certification" in text_lower
    }

    score = (
        len(found) / len(keywords) * 70
        +
        sum(sections.values()) / len(sections) * 30
    )

    print(f"\n📊 Resume Score: {score:.1f}/100")

    print(
        "\n✅ Detected Skills:",
        ", ".join(found)
    )

    print(
        "\n📚 Suggested Additions:",
        ", ".join(missing[:10])
    )

    print("\n📌 Resume Sections:")

    for section, exists in sections.items():

        print(
            f"{'✅' if exists else '❌'} {section.title()}"
        )


# ================================================================
# 14. CAREER MISSION
# ================================================================

def career_mission():

    profile = get_profile()

    if not profile:
        print("⚠️ Create Student Profile first.")
        return

    role = profile["target_role"]

    if not role:
        role = "AI/ML Engineer"

    print("\n🚀 CAREER MISSION")
    print("=" * 60)

    print(f"🎯 Target: {role}")

    missions = [
        "Complete one core technical course.",
        "Build one practical AI/ML project.",
        "Push the project to GitHub.",
        "Improve your resume.",
        "Apply to 5 relevant internships.",
        "Practice 5 interview questions.",
        "Learn one missing skill.",
        "Complete one certification or workshop."
    ]

    for i, mission in enumerate(
        missions, 1
    ):
        print(f"{i}. {mission}")

    print(
        "\n🏆 Goal: Build skills + projects + experience "
        "instead of only collecting certificates."
    )


# ================================================================
# 15. CAREER DIGITAL TWIN
# ================================================================

def digital_twin():

    profile = get_profile()

    if not profile:
        print("⚠️ Create Student Profile first.")
        return

    print("\n👤 CAREER DIGITAL TWIN")
    print("=" * 60)

    print(
        f"Student: {profile['name']}"
    )

    print(
        f"Target Role: {profile['target_role']}"
    )

    skills = profile["skills"]

    print(
        "\n🧠 Current Skill Profile:"
    )

    for skill in skills:
        print(" •", skill)

    print(
        "\n📈 Digital Twin Assessment:"
    )

    if len(skills) >= 8:
        print("🟢 Strong technical profile")
    elif len(skills) >= 5:
        print("🟡 Developing technical profile")
    else:
        print("🔴 Beginner profile — build more skills/projects")

    print(
        "\nRecommended next step:"
    )

    print(
        "Build projects that directly demonstrate "
        "your target-role skills."
    )


# ================================================================
# 16. WHAT-IF CAREER LAB
# ================================================================

def what_if_lab():

    profile = get_profile()

    if not profile:
        print("⚠️ Create Student Profile first.")
        return

    print("\n🔮 WHAT-IF CAREER LAB")
    print("=" * 60)

    role = input(
        "Target career: "
    ).strip()

    current = set(
        normalize(x)
        for x in profile["skills"]
    )

    additional_text = input(
        "Skills you plan to add (comma separated): "
    )

    additional = set(
        normalize(x.strip())
        for x in additional_text.split(",")
        if x.strip()
    )

    future = current.union(additional)

    if role in CAREERS:

        required = set(
            normalize(x)
            for x in CAREERS[role]["skills"]
        )

        before = (
            len(current.intersection(required))
            / len(required) * 100
        )

        after = (
            len(future.intersection(required))
            / len(required) * 100
        )

        print(
            f"\n📊 Current Readiness: {before:.1f}%"
        )

        print(
            f"🚀 Future Readiness: {after:.1f}%"
        )

        print(
            f"📈 Improvement: {after-before:+.1f}%"
        )

        gaps = required - future

        print(
            "\n📚 Remaining Skill Gaps:",
            ", ".join(gaps) or "None"
        )

    else:

        print(
            "⚠️ Career not found in database."
        )


# ================================================================
# 17. INTERVIEW SIMULATOR
# ================================================================

INTERVIEW_QUESTIONS = {

    "AI/ML Engineer": [
        "What is Machine Learning?",
        "What is overfitting and how can you prevent it?",
        "What is feature engineering?",
        "How would you deploy a machine learning model?",
        "What is Generative AI?",
        "What is NLP?",
        "What is the difference between classification and regression?"
    ],

    "Software Developer": [
        "What is OOP?",
        "What is inheritance?",
        "What is a database?",
        "What is an API?",
        "What is Git?",
        "What is the difference between a list and tuple in Python?"
    ],

    "Data Scientist": [
        "What is data preprocessing?",
        "What is correlation?",
        "What is hypothesis testing?",
        "What is feature engineering?",
        "What is cross-validation?"
    ],

    "Cybersecurity Analyst": [
        "What is phishing?",
        "What is malware?",
        "What is a firewall?",
        "What is encryption?",
        "What is multi-factor authentication?"
    ]
}


def interview_simulator():

    print("\n🎤 INTERVIEW SIMULATOR")
    print("=" * 60)

    role = input(
        "Enter interview role: "
    ).strip()

    questions = INTERVIEW_QUESTIONS.get(
        role,
        INTERVIEW_QUESTIONS["AI/ML Engineer"]
    )

    score = 0

    for i, question in enumerate(
        questions,
        1
    ):

        print(
            f"\nQuestion {i}/{len(questions)}"
        )

        print("🎤", question)

        answer = input(
            "\nYour answer: "
        ).strip()

        if len(answer) < 20:

            print(
                "⚠️ Answer is too short."
            )

            continue

        keywords = {
            "machine learning":
                ["data", "model", "learn"],

            "generative ai":
                ["generate", "content", "model"],

            "nlp":
                ["language", "text", "human"],

            "overfitting":
                ["training", "validation", "regularization"],

            "feature engineering":
                ["feature", "data", "model"],

            "deploy":
                ["server", "api", "cloud", "model"]
        }

        q_lower = question.lower()

        matched = 0

        for key, words in keywords.items():

            if key in q_lower:

                matched = sum(
                    word in answer.lower()
                    for word in words
                )

                break

        if matched >= 2:
            score += 10
            print("✅ Good answer.")
        elif matched == 1:
            score += 6
            print("🟡 Partially correct.")
        else:
            score += 3
            print(
                "🔴 Add more technical explanation."
            )

    print("\n" + "=" * 60)
    print("🎤 INTERVIEW RESULT")
    print("=" * 60)

    print(
        f"Score: {score}/{len(questions)*10}"
    )

    percentage = (
        score /
        (len(questions) * 10)
        * 100
    )

    print(
        f"Performance: {percentage:.1f}%"
    )


# ================================================================
# 18. APPLICATION TRACKING
# ================================================================

def add_application():

    print("\n📋 ADD APPLICATION")
    print("=" * 60)

    company = input("Company: ")
    role = input("Role: ")
    url = input("Job URL: ")
    status = input(
        "Status [Applied/Interview/Rejected/Selected]: "
    )
    notes = input("Notes: ")

    cursor.execute("""
    INSERT INTO applications
    (
        user_id, company, role,
        url, status, notes, applied_date
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        CURRENT_USER["id"],
        company,
        role,
        url,
        status,
        notes,
        datetime.now().strftime("%Y-%m-%d")
    ))

    conn.commit()

    print("✅ Application saved.")


def view_applications():

    print("\n📋 APPLICATION TRACKER")
    print("=" * 70)

    if CURRENT_USER["id"] == 0:

        print(
            "Demo user cannot permanently store applications."
        )

        return

    cursor.execute("""
    SELECT id, company, role,
           status, applied_date, url
    FROM applications
    WHERE user_id=?
    ORDER BY id DESC
    """, (
        CURRENT_USER["id"],
    ))

    rows = cursor.fetchall()

    if not rows:

        print("No applications yet.")
        return

    df = pd.DataFrame(
        rows,
        columns=[
            "ID",
            "Company",
            "Role",
            "Status",
            "Applied Date",
            "URL"
        ]
    )

    print(df.to_string(index=False))


# ================================================================
# 19. CAREER DASHBOARD
# ================================================================

def dashboard():

    profile = get_profile()

    print("\n" + "=" * 70)
    print("📊 CAREERPILOT-X 2.0 DASHBOARD")
    print("=" * 70)

    print(
        f"👤 User: {CURRENT_USER['username']}"
    )

    if profile:

        print(
            f"🎓 Student: {profile['name']}"
        )

        print(
            f"🎯 Target: {profile['target_role']}"
        )

        print(
            f"🧠 Skills: {len(profile['skills'])}"
        )

    else:

        print(
            "⚠️ Student profile not created yet."
        )

    if CURRENT_USER["id"] != 0:

        cursor.execute(
            "SELECT COUNT(*) FROM applications WHERE user_id=?",
            (CURRENT_USER["id"],)
        )

        count = cursor.fetchone()[0]

        print(
            f"📋 Applications: {count}"
        )

    print("\n🚀 Available Intelligence Modules:")
    print(" • Career Intelligence")
    print(" • Verified Job Search")
    print(" • Fake Job Detection")
    print(" • Opportunity DNA")
    print(" • Skill Matching")
    print(" • Resume Analyzer")
    print(" • Career Mission")
    print(" • Digital Twin")
    print(" • What-If Lab")
    print(" • Interview Simulator")
    print(" • Application Tracking")


# ================================================================
# 20. SECURITY CENTER
# ================================================================

def security_center():

    print("\n🔒 SECURITY CENTER")
    print("=" * 60)

    print("✅ Password hashing enabled")
    print("✅ User-specific profiles")
    print("✅ User-specific applications")
    print("✅ SQLite database")
    print("✅ URL risk analysis")
    print("✅ Fake job detection")
    print("✅ Scam-risk scoring")
    print("✅ Sensitive information warnings")

    print(
        "\n⚠️ Safety rule:"
    )

    print(
        "Never share OTP, CVV, PIN, passwords "
        "or banking credentials with a recruiter."
    )


# ================================================================
# 21. MAIN MENU
# ================================================================

def main_menu():

    while True:

        print("\n" + "=" * 70)
        print("🚀 CAREERPILOT-X 2.0")
        print("=" * 70)

        print("1. 📊 Career Dashboard")
        print("2. 👩‍🎓 Student Profile")
        print("3. 🧠 Career Intelligence")
        print("4. 🔎 Verified Job Search")
        print("5. 🛡️ Fake Job Detection")
        print("6. 🧬 Opportunity DNA")
        print("7. 🎯 Skill Matching")
        print("8. 📄 Resume Analyzer")
        print("9. 🚀 Career Mission")
        print("10. 👤 Career Digital Twin")
        print("11. 🔮 What-If Career Lab")
        print("12. 🎤 Interview Simulator")
        print("13. 📋 Add Application")
        print("14. 📋 View Applications")
        print("15. 🔒 Security Center")
        print("16. 🔑 Configure SerpApi")
        print("17. 🚪 Logout")

        choice = input(
            "\nChoose an option: "
        ).strip()

        if choice == "1":
            dashboard()

        elif choice == "2":
            student_profile()

        elif choice == "3":
            career_intelligence()

        elif choice == "4":
            search_jobs()

        elif choice == "5":
            fake_job_detector()

        elif choice == "6":
            opportunity_dna()

        elif choice == "7":
            skill_matching()

        elif choice == "8":
            resume_analyzer()

        elif choice == "9":
            career_mission()

        elif choice == "10":
            digital_twin()

        elif choice == "11":
            what_if_lab()

        elif choice == "12":
            interview_simulator()

        elif choice == "13":
            add_application()

        elif choice == "14":
            view_applications()

        elif choice == "15":
            security_center()

        elif choice == "16":
            set_serpapi_key()

        elif choice == "17":

            print(
                "\n👋 Logged out of CareerPilot-X."
            )

            break

        else:

            print(
                "❌ Invalid option."
            )


# ================================================================
# 22. START APPLICATION
# ================================================================

if authentication():

    main_menu()

else:

    print(
        "CareerPilot-X terminated."
    )

🚀 CAREERPILOT-X 2.0
📁 Data directory: /content/careerpilot_data
✅ Storage initialized


🔐 CAREERPILOT-X SECURITY
1. Login
2. Create Account
3. Continue as Demo User
4. Exit

Choose: 1

🔐 LOGIN
----------------------------------------
Username: Sravani Viswanadhapalli
Password: ··········

✅ Welcome back, Sravani Viswanadhapalli!

🚀 CAREERPILOT-X 2.0
1. 📊 Career Dashboard
2. 👩‍🎓 Student Profile
3. 🧠 Career Intelligence
4. 🔎 Verified Job Search
5. 🛡️ Fake Job Detection
6. 🧬 Opportunity DNA
7. 🎯 Skill Matching
8. 📄 Resume Analyzer
9. 🚀 Career Mission
10. 👤 Career Digital Twin
11. 🔮 What-If Career Lab
12. 🎤 Interview Simulator
13. 📋 Add Application
14. 📋 View Applications
15. 🔒 Security Center
16. 🔑 Configure SerpApi
17. 🚪 Logout

Choose an option: 9

🚀 CAREER MISSION
🎯 Target: ai engineer
1. Complete one core technical course.
2. Build one practical AI/ML project.
3. Push the project to GitHub.
4. Improve your resume.
5. Apply to 5 relevant internships.
6. Practice 5 interview questions.
